In [ ]:
import json
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
import plotly.express as px
import streamlit as st
from pathlib import Path

st.set_page_config(page_title="Key US ZIPs + Trader Joe's Impact Analysis", layout="wide")

# --- File paths (MUST be updated to point to US-WIDE datasets) ---
BASE = Path(__file__).resolve().parents[1]

# ⚠️ ACTION NEEDED: Update these paths to your US-wide data files
# Assuming a single file containing all US Trader Joe's locations
TJ_PATH = BASE / "data" / "raw" / "us-tj-locations.csv"

# Assuming a single file containing all US ZCTA polygons with predicted prices
PARQUET_PATH = BASE / "data" / "processed_data" / "us_zcta_with_prices.parquet"


# --- Load tabular data (US-Wide Logic) ---
@st.cache_data
def load_zcta(path: Path) -> gpd.GeoDataFrame:
    """Loads and preprocesses US ZCTA GeoDataFrame."""
    try:
        gdf = gpd.read_parquet(path)
    except Exception as e:
        st.error(f"Error loading US ZCTA data from {path}. Check file path and format. ({e})")
        st.stop()
        
    if 'state' not in gdf.columns:
         st.error("Missing 'state' column in ZCTA data. Cannot filter by state.")
         st.stop()
         
    if 'zip' in gdf.columns:
        gdf['zip'] = gdf['zip'].astype(str).str.zfill(5)
        
    # Standardize state names (if your data uses full names)
    # We will assume your 'state' column uses two-letter abbreviations (e.g., 'CA', 'TX') 
    # and adjust the filtering list below.
    return gdf

@st.cache_data
def load_tj(path: Path) -> pd.DataFrame:
    """Loads US Trader Joe's locations."""
    try:
        df = pd.read_csv(path)
    except Exception as e:
        st.error(f"Error loading US TJ locations from {path}. Check file path and format. ({e})")
        st.stop()

    df["state"] = df["state"].str.upper() # Standardize state codes
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    return df

# Load all data initially
zcta_all = load_zcta(PARQUET_PATH)
tjs_all = load_tj(TJ_PATH)

# --- NEW: Define the specific states to include ---
# You need to ensure these match the state identifier format (e.g., full name or abbreviation) 
# used in your actual data files (zcta_all['state'] and tjs_all['state']).
# ASSUMPTION: The data uses the two-letter abbreviation for filtering.
# If your data uses the full state names, replace these abbreviations with the full names.
STATE_ABBREVIATION_MAP = {
    "California": "CA", 
    "Texas": "TX", 
    "Florida": "FL", 
    "Washington": "WA", 
    "New York": "NY"
}
# The list of states shown to the user in the filter
ALL_STATES_NAMES = list(STATE_ABBREVIATION_MAP.keys())
# The list of abbreviations used for filtering the data
FILTER_ABBREVIATIONS = list(STATE_ABBREVIATION_MAP.values())


# --- Filters Section ---
st.sidebar.header("Map Filters")

# 1. State Filter 
selected_states_names = st.sidebar.multiselect(
    "Select State(s):",
    options=ALL_STATES_NAMES,
    # Default to all five specified states
    default=ALL_STATES_NAMES
)

if not selected_states_names:
    st.info("Please select one or more states to view the data.")
    st.stop()

# Convert selected names back to abbreviations for filtering
selected_abbreviations = [STATE_ABBREVIATION_MAP[name] for name in selected_states_names]

# Filter data based on selection
zcta_filtered = zcta_all[zcta_all['state'].isin(selected_abbreviations)]
tjs_filtered = tjs_all[tjs_all['state'].isin(selected_abbreviations)]


# 2. Color Selection (with better aliases)
COLOR_OPTIONS = {
    "Median Predicted Price": "pred_price_median",
    "Max Predicted Price": "pred_price_max",
    "Min Predicted Price": "pred_price_min",
    "Population Density": "density"
}

color_display_name = st.sidebar.selectbox(
    "Color ZIPs by:",
    options=list(COLOR_OPTIONS.keys()),
    index=0
)
color_col = COLOR_OPTIONS[color_display_name]


# Apply final filtering and prepare GeoJSON
visible = zcta_filtered[~zcta_filtered[color_col].isna()]

if visible.empty:
    st.warning("No data found for the selected state(s) and metric.")
    st.stop()

geojson = json.loads(visible.to_json())

# --- View Section ---
# Dynamic Title and Overview based on selection
state_list_str = ", ".join(selected_states_names) if len(selected_states_names) <= 3 else f"{len(selected_states_names)} Key States"
st.title(f"ZIP Code Boundaries & Trader Joe’s Analysis for {state_list_str}")
st.markdown(
    """
    **Project Overview**: Our project investigates the association between **housing prices** and **proximity to Trader Joe's locations** by using a predictive model.
    """
)
st.markdown(
    """
    **Data Overview**: The visualization displays **ZIP Code Tabulation Areas (ZCTAs)** colored by a **predicted housing price metric** and overlaid with **Trader Joe's store locations**.
    """
)

# ==============================================================================
# STATE OVERVIEW OVERLAY CARD (CSS must be present in the final app)
# ==============================================================================

# Card appears only if a single state is selected
if len(selected_states_names) == 1:
    state_name = selected_states_names[0]
    
    # 1. CALCULATE SUMMARY METRICS
    total_tjs = len(tjs_filtered)
    avg_pop_density = visible["density"].mean()
    selected_metric_value = visible[color_col].median() 
    
    # Format the metrics
    total_tjs_str = f"{total_tjs:,}"
    avg_pop_density_str = f"{avg_pop_density:,.0f}"
    selected_metric_str = f"${selected_metric_value:,.0f}"
    
    # Note: The CSS for .fixed-card and .card-metric must be included 
    # (from a previous revision) for this HTML to render correctly.
    
    overlay_card_html = f"""
    <div class="fixed-card">
        <h4 style="margin: 0 0 10px 0;">State Overview: **{state_name}**</h4>
        <div class="card-metric">
            <span class="metric-label">Predicted **{color_display_name}**:</span>
            <span class="metric-value">{selected_metric_str} (Median)</span>
        </div>
        <div class="card-metric">
            <span class="metric-label">Average Population Density:</span>
            <span class="metric-value">{avg_pop_density_str} ppl/mi²</span>
        </div>
        <div class="card-metric">
            <span class="metric-label">Number of Trader Joe's:</span>
            <span class="metric-value">{total_tjs_str}</span>
        </div>
    </div>
    """
    st.markdown(overlay_card_html, unsafe_allow_html=True)
# ==============================================================================


# --- Create Plotly figure with polygons ---
# Calculate the center of the selection for better map focus
center_lat = visible["latitude"].mean()
center_lon = visible["longitude"].mean()

fig = px.choropleth_mapbox(
    visible,
    geojson=geojson,
    locations="zip",
    featureidkey="properties.zip",
    color=color_col,
    color_continuous_scale="Viridis",
    hover_name="zip",
    hover_data={
        "city": True,
        "county_name": True,
        "population": True,
        "density": True,
        "pred_price_median": ":$,.0f",
        "pred_price_mean": ":$,.0f",
        "pred_price_max": ":$,.0f",
        "zip": False
    },
    labels={
        "pred_price_median": "Median Pred Price ($)",
        "pred_price_max": "Max Pred Price ($)",
        "pred_price_min": "Min Pred Price ($)",
        "density": "Density (ppl/mi²)"
    },
    mapbox_style="carto-positron",
    center={"lat": center_lat, "lon": center_lon},
    # Adjust zoom level for a regional view of these spread out states
    zoom=2.5 if len(selected_states_names) > 3 else (4.5 if len(selected_states_names) > 1 else 6),
    opacity=0.7
)


# --- State boundary overlay ---
boundary_gdf = gpd.GeoDataFrame(
    {'group': ['selected_area']},
    geometry=[zcta_filtered.geometry.unary_union],
    crs=zcta_filtered.crs
)
boundary_geojson = json.loads(boundary_gdf.to_json())

fig.add_trace(
    go.Choroplethmapbox(
        geojson=boundary_geojson,
        locations=boundary_gdf['group'],
        featureidkey="properties.group",
        z=[1],
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],
        showscale=False,
        marker_opacity=1,
        marker_line_width=2,
        marker_line_color='black',
        name='Selected Boundary',
        hoverinfo='skip'
    )
)

# --- Add Trader Joe's point layer ---
fig.add_trace(
    go.Scattermapbox(
        lat=tjs_filtered["latitude"],
        lon=tjs_filtered["longitude"],
        mode="markers",
        marker=dict(
            size=10,
            color="red",
            opacity=0.8,
            symbol="circle"
        ),
        name="Trader Joe's",
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "%{customdata[1]}<br>"
            "%{customdata[2]}, %{customdata[3]} %{customdata[4]}<extra></extra>"
        ),
        customdata=tjs_filtered[["name","street","city","state","zip"]].values
    )
)

fig.update_layout(
    margin={"r":0,"t":0,"l":0,"b":0},
    coloraxis_colorbar_title_text=color_display_name
)

st.plotly_chart(fig, use_container_width=True)

# --- Findings and Analysis Section ---
st.subheader(f"Key Findings")
st.info(
    """
    **Naz's Analysis (Example)**:
    - Each additional mile farther from the nearest Trader Joe’s is associated with approximately a **2.5–2.7% decrease in home prices**, holding other features constant.
    - **ZIP Codes in proximity to a Trader Joe's store** tend to have median predicted housing prices in the top **20%** of the analyzed regions.
    """
)
st.caption(
    "Data Source Notes: Requires US-wide ZCTA boundaries with predicted prices and US-wide Trader Joe’s locations. Filtered to key states: CA, TX, FL, WA, NY."
)